In [1]:
import pandas as pd

In [3]:
df6 = pd.read_csv(r".\log\step2.csv")
df6 = df6.drop(['Unnamed: 0'], axis=1, errors='ignore')
df6

,frequency,355.26607692224155,227.52736891479316,228.1048056598934,508.4579820682637,624.7739429956808,509.2412520609261,746.6584998199332,738.8342325716347,744.5957020753282,...,854.7166023522125,858.2383815524672,851.444148653565,887.258212048309,912.2418726871764,911.7194528327119,1047.003667471031,1047.5534428923963,1000.9177192045671,Max_Oxygen_LifeTime
0,0.2,15.693691,9.443991,1.249940,0.000000,34.094721,0.000000,6.857589,167.325163,5.486071,...,18.577831,0.000000,0.000000,0.000000,386.246099,0.000000,15.371201,12.332707,5.183312,0.459737
1,0.2,0.000000,14.682523,0.000000,0.000000,103.661344,0.000000,31.635450,318.990789,0.000000,...,87.348993,0.000000,43.410868,0.000000,756.836451,1.259714,29.981204,34.012291,0.000000,0.971616
2,0.2,6.163256,29.393991,0.000000,46.937403,237.872811,100.458785,12.768803,501.175525,0.000000,...,58.736495,0.000000,120.452377,0.000000,1054.932155,15.863641,0.000000,12.812941,0.000000,0.911778
3,0.2,0.000000,13.837251,53.544146,0.000000,189.468595,114.004574,47.529882,645.434196,0.000000,...,120.985155,109.912853,163.654026,0.000000,1481.539146,25.550453,54.197931,33.293015,0.000000,0.939168
4,0.2,0.000000,22.551184,0.000000,8.134488,158.228915,106.273152,34.180944,740.674757,0.000000,...,67.573096,0.000000,180.896071,0.000000,1739.831124,12.438134,41.837360,36.183663,11.307395,0.841546
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
574,1.0,11.021610,0.000000,0.000000,0.000000,15.287786,98.902616,107.145214,952.543208,191.080931,...,0.000000,0.000000,348.778338,68.356739,2267.961964,36.573251,23.226117,12.085988,2.627389,0.738645
575,1.0,16.868346,0.000000,0.000000,39.158921,0.000000,189.268120,92.778647,948.403947,120.379463,...,62.517511,112.731044,259.713704,122.042163,2514.359653,26.381320,0.000000,18.796691,15.718870,0.683652
576,1.0,21.019440,0.000000,0.000000,0.000000,0.000000,40.558919,0.000000,948.886917,49.319891,...,0.000000,0.000000,248.336071,81.273623,2804.664412,32.490893,38.805378,32.950128,15.154763,0.652843
577,1.0,32.316066,0.000000,0.000000,0.000000,0.000000,98.823214,144.473708,1428.856588,38.345568,...,0.000000,158.804880,436.519754,0.000000,3477.884529,56.334457,24.070177,22.405750,7.553939,0.622883


In [ ]:
import numpy as np
import pandas as pd
import warnings
import os
import joblib
warnings.filterwarnings("ignore")

from sklearn.model_selection import (
    train_test_split, RepeatedKFold, RandomizedSearchCV, cross_val_score
)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline

RANDOM_STATE = 4

# -----------------------------
# 0. Data Preparation
# -----------------------------
X = df6.drop('Max_Oxygen_LifeTime', axis=1).values
y = df6['Max_Oxygen_LifeTime'].values.ravel()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=RANDOM_STATE
)

# Scaler for the search phase (only for Train data)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Equalize the folds for both models (5×3 = 15)
cv_search = RepeatedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE)
cv_eval = RepeatedKFold(n_splits=5, n_repeats=20, random_state=RANDOM_STATE)

# -----------------------------
# Evaluation function without data leakage (with Pipeline)
# -----------------------------
def evaluate_model(name, model, X_tr, X_te, y_tr, y_te, X_full, y_full):
    # Build a pipeline to ensure proper scaling in CV
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('regressor', model)
    ])
    
    pipeline.fit(X_tr, y_tr)
    y_tr_pred = pipeline.predict(X_tr)
    y_te_pred = pipeline.predict(X_te)

    rmse_train = np.sqrt(mean_squared_error(y_tr, y_tr_pred))
    rmse_test = np.sqrt(mean_squared_error(y_te, y_te_pred))
    r2_train = r2_score(y_tr, y_tr_pred)
    r2_test = r2_score(y_te, y_te_pred)

   # Cross-Validation with pipeline (separate scaling in each fold)
    cv_scores = cross_val_score(
        pipeline, X_full, y_full, cv=cv_eval, scoring='r2', n_jobs=-1
    )

    print(f"\n===== {name} =====")
    print(f"RMSE Train: {rmse_train:.4f} | RMSE Test: {rmse_test:.4f}")
    print(f"R² Train:   {r2_train:.4f} | R² Test:   {r2_test:.4f}")
    print(f"CV R² (mean ± std, {cv_eval.get_n_splits()} folds): "
          f"{cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

    return {
        "model": name, "rmse_train": rmse_train, "rmse_test": rmse_test,
        "r2_train": r2_train, "r2_test": r2_test,
        "cv_mean": cv_scores.mean(), "cv_std": cv_scores.std(),
    }

# -----------------------------
#1. Search space
# -----------------------------
rf_param_grid = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [3, 5, 7, 10, None],
    "min_samples_split": [2, 4, 6, 10],
    "min_samples_leaf": [1, 2, 4, 6],
    "max_features": ["sqrt", "log2", 0.5, 0.7, 1.0],
}

# -----------------------------
# 2. Random search instead of comprehensive search (more efficient)
# -----------------------------
print("Starting Randomized Search for RF...")
rf_random_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions=rf_param_grid,
    n_iter=150,  # Number of random combinations (sufficient and standard for paper ones)
    cv=cv_search,
    scoring='r2',
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=2
)
rf_random_search.fit(X_train_scaled, y_train)

print("\nBest parameters found:")
print(rf_random_search.best_params_)
print(f"Best R² score (on validation): {rf_random_search.best_score_:.4f}")

# -----------------------------
#3. Final evaluation (with raw data for the pipeline)
# -----------------------------
result = evaluate_model(
    "Random Forest (RandomizedSearchCV)",
    rf_random_search.best_estimator_,
    X_train, X_test, y_train, y_test, X, y  # Raw data is sent
)

print("\nFinal result:")
print(result)

# -----------------------------
#4. Save the model and scaler
# -----------------------------
os.makedirs("RandomForestModel2", exist_ok=True)
joblib.dump(rf_random_search.best_estimator_, "RandomForestModel2/best_model.pkl")
joblib.dump(scaler, "RandomForestModel2/scaler.pkl")
print("\nModel and scaler saved in 'RandomForestModel2' folder.")

Starting Randomized Search for RF...
Fitting 15 folds for each of 150 candidates, totalling 2250 fits

Best parameters found:
{'n_estimators': 500, 'min_samples_split': 4, 'min_samples_leaf': 2, 'max_features': 1.0, 'max_depth': 10}
Best R² score (on validation): 0.3630

===== Random Forest (RandomizedSearchCV) =====
RMSE Train: 0.0375 | RMSE Test: 0.0544
R² Train:   0.8500 | R² Test:   0.6690
CV R² (mean ± std, 100 folds): 0.3802 ± 0.0881

Final result:
{'model': 'Random Forest (RandomizedSearchCV)', 'rmse_train': np.float64(0.03752587568504041), 'rmse_test': np.float64(0.054426027709888636), 'r2_train': 0.8499636225804913, 'r2_test': 0.6690476315867586, 'cv_mean': np.float64(0.38019040669863424), 'cv_std': np.float64(0.08807409705268261)}

Model and scaler saved in 'RandomForestModel2' folder.


In [ ]:
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import (
    train_test_split, RepeatedKFold, RandomizedSearchCV, cross_val_score
)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline 

RANDOM_STATE = 4

# -----------------------------
# 0. Data Preparation
# -----------------------------
X = df6.drop('Max_Oxygen_LifeTime', axis=1).values
y = df6['Max_Oxygen_LifeTime'].values.ravel()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=RANDOM_STATE
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Unify the folds with Random Forest code (5×3 = 15)
cv_search = RepeatedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE)
cv_eval = RepeatedKFold(n_splits=5, n_repeats=20, random_state=RANDOM_STATE)

# -----------------------------
# Evaluation function without data leakage (exactly the same as RF code)
# -----------------------------
def evaluate_model(name, model, X_tr, X_te, y_tr, y_te, X_full, y_full):
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('regressor', model)
    ])
    
    pipeline.fit(X_tr, y_tr)
    y_tr_pred = pipeline.predict(X_tr)
    y_te_pred = pipeline.predict(X_te)

    rmse_train = np.sqrt(mean_squared_error(y_tr, y_tr_pred))
    rmse_test = np.sqrt(mean_squared_error(y_te, y_te_pred))
    r2_train = r2_score(y_tr, y_tr_pred)
    r2_test = r2_score(y_te, y_te_pred)

    cv_scores = cross_val_score(
        pipeline, X_full, y_full, cv=cv_eval, scoring='r2', n_jobs=-1
    )

    print(f"\n===== {name} =====")
    print(f"RMSE Train: {rmse_train:.4f} | RMSE Test: {rmse_test:.4f}")
    print(f"R² Train:   {r2_train:.4f} | R² Test:   {r2_test:.4f}")
    print(f"CV R² (mean ± std, {cv_eval.get_n_splits()} folds): "
          f"{cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

    return {
        "model": name, "rmse_train": rmse_train, "rmse_test": rmse_test,
        "r2_train": r2_train, "r2_test": r2_test,
        "cv_mean": cv_scores.mean(), "cv_std": cv_scores.std(),
    }

# -----------------------------
#1. Search space
# -----------------------------
xgb_param_grid = {
    'n_estimators': [100, 200, 300, 500],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 5, 7, 10],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'reg_alpha': [0, 0.01, 0.1, 1],
    'reg_lambda': [0.1, 0.5, 1.0, 2.0],
    'min_child_weight': [0.1, 0.4, 1, 3],
}

# -----------------------------
#2. Random Search (alternative to GridSearch to avoid 400k fitting times)
# -----------------------------
print("Starting Randomized Search for XGBoost...")
xgb_random_search = RandomizedSearchCV(
    XGBRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions=xgb_param_grid,
    n_iter=100,  # Only 100 random combinations
    cv=cv_search,
    scoring='r2',
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=2
)
xgb_random_search.fit(X_train_scaled, y_train)

print("\nBest parameters found:")
print(xgb_random_search.best_params_)
print(f"Best R² score (validation): {xgb_random_search.best_score_:.4f}")

# -----------------------------
#3. Final evaluation with raw data
# -----------------------------
result = evaluate_model(
    "XGBoost (RandomizedSearchCV)",
    xgb_random_search.best_estimator_,
    X_train, X_test, y_train, y_test, X, y 
)
print("\nFinal Result:")
print(result)

os.makedirs("XGBoostModel23", exist_ok=True)
joblib.dump(xgb_random_search.best_estimator_, "XGBoostModel23/best_model.pkl")
joblib.dump(scaler, "XGBoostModel23/scaler.pkl")

Starting Randomized Search for XGBoost...
Fitting 15 folds for each of 100 candidates, totalling 1500 fits

Best parameters found:
{'subsample': 0.7, 'reg_lambda': 1.0, 'reg_alpha': 0, 'n_estimators': 300, 'min_child_weight': 1, 'max_depth': 10, 'learning_rate': 0.01, 'colsample_bytree': 0.8}
Best R² score (validation): 0.3813

===== XGBoost (RandomizedSearchCV) =====
RMSE Train: 0.0218 | RMSE Test: 0.0539
R² Train:   0.9496 | R² Test:   0.6753
CV R² (mean ± std, 100 folds): 0.3991 ± 0.0775

Final Result:
{'model': 'XGBoost (RandomizedSearchCV)', 'rmse_train': np.float64(0.021758986593178403), 'rmse_test': np.float64(0.053912668598075114), 'r2_train': 0.949555685130313, 'r2_test': 0.675261429095238, 'cv_mean': np.float64(0.3990584233604742), 'cv_std': np.float64(0.07752132247893138)}


['XGBoostModel23/scaler.pkl']